# local-wlm — GPU render on Kaggle T4
Clones the repo, runs the Genesis physics sims on the free GPU, writes 1080p MP4s to the notebook output.

In [ ]:
!nvidia-smi -L
import os
# headless GPU rendering (no display on Kaggle)
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MUJOCO_GL"] = "egl"

In [ ]:
!pip install -q genesis-world 2>&1 | tail -3
# Kaggle's default torch (2.10) dropped support for the P100 GPU (sm_60) it often
# assigns on the free tier. Install a torch with kernels for P100 AND T4 (sm_75).
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -2
!rm -rf local-wlm && git clone -q https://github.com/varunkar2003/local-wlm.git
%cd local-wlm
# confirm torch sees the GPU with a usable kernel image
import torch
print("torch", torch.__version__, "| cuda ok:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

In [ ]:
# the headline shot: mesh humanoid swimming, dense water, 1080p
!python sims/swimmer_g1.py --gpu --particle-size 0.015 --res 1920 1080 --seconds 5

In [ ]:
!python sims/dominoes.py --gpu --res 1920 1080
!python sims/water_splash.py --gpu --particle-size 0.008 --res 1920 1080

In [ ]:
# expose results in the kernel output
import shutil, glob, os
os.makedirs("/kaggle/working", exist_ok=True)
for f in glob.glob("out/*.mp4"):
    shutil.copy(f, "/kaggle/working/")
print(os.listdir("/kaggle/working"))